# P1 v6 — 2-Group Severity Adapters + Soft MLP Routing

## Architecture

**Training:**
- B2 backbone: fully frozen
- 2 LoRA adapters + 2 CTC heads, initialised from LibriSpeech CTC weights
  - Group 0 (`mild_ctrl`): trains on control + mild utterances
  - Group 1 (`mod_sev`):   trains on moderate + severe utterances
- LibriSpeech init: higher initial loss → stronger gradients → more adaptation

**Inference:**
```
Audio → B2 encoder (frozen)
    ↓                              ↓
adapter_0 → CTC head_0      Layer-6 → Severity MLP (pretrained, 4-class)
adapter_1 → CTC head_1           ↓
    ↓                    p = [p_ctrl, p_mild, p_mod, p_sev]
logits_0, logits_1               ↓
    ↓               w_0 = p_ctrl + p_mild   (mild_ctrl group weight)
    ↓               w_1 = p_mod  + p_sev    (mod_sev group weight)
    ↓
final_logits = w_0 × logits_0 + w_1 × logits_1
    ↓
KenLM beam search
```

In [1]:
import os, warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("pyctcdecode").setLevel(logging.ERROR)
os.environ["CUDA_VISIBLE_DEVICES"]      = "0"
os.environ["TOKENIZERS_PARALLELISM"]    = "false"
os.environ["HF_HOME"]                   = "/kaggle/temp/hf"
os.environ["HF_DATASETS_CACHE"]         = "/kaggle/temp/hf/datasets"
os.environ["HF_HUB_CACHE"]              = "/kaggle/temp/hf/hub"
os.environ["TRANSFORMERS_CACHE"]        = "/kaggle/temp/hf/transformers"
os.environ["XDG_CACHE_HOME"]            = "/kaggle/temp/.cache"
for p in [os.environ["HF_HOME"], os.environ["HF_DATASETS_CACHE"],
           os.environ["HF_HUB_CACHE"], os.environ["TRANSFORMERS_CACHE"],
           os.environ["XDG_CACHE_HOME"]]:
    os.makedirs(p, exist_ok=True)
print("Env ready.")

Env ready.


!pip install pyctcdecode==0.5.0
!pip install kenlm
!pip install numpy==1.26.4
!pip -q install -U transformers datasets evaluate jiwer soundfile huggingface_hub scikit-optimize
!pip -q install safetensors

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import evaluate
import json
import random
import sys
from itertools import groupby
from collections import Counter
from typing import Optional

from datasets import load_dataset
from transformers import WavLMForCTC, Wav2Vec2Processor
from pyctcdecode import build_ctcdecoder
from huggingface_hub import hf_hub_download

print("torch:", torch.__version__)
print("GPU:",   torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
GPU: True
GPU: Tesla T4


## Step 1 — Config

In [3]:
B2_PATH        = "/kaggle/input/datasets/kavishk/b2-pretrain/kaggle/working/b2v2_wavlm_ctc_final"
MLP_PT         = "/kaggle/input/datasets/kavishk/severity-mlp/p2_severity_mlp.pt"

RANDOM_SEED    = 42
TEST_SPEAKERS  = {"F01", "F04", "FC01", "M05"}
MLP_VAL_FRAC   = 0.2
CONTROL_TARGET = 1500
MAX_AUDIO      = 240_000
NUM_SEV_CLASSES = 4
NUM_GROUPS      = 2

TORGO_SEV_TO_INT = {"control": 0, "mild": 1, "moderate": 2, "severe": 3}
INT_TO_SEV       = {v: k for k, v in TORGO_SEV_TO_INT.items()}
SEVERITY_MAPPING = {
    "F01": "severe",   "M01": "severe",   "M02": "severe",   "M04": "severe",
    "M05": "moderate", "F03": "moderate",
    "F04": "mild",     "M03": "mild",
    "FC01": "control", "FC02": "control", "FC03": "control",
    "MC01": "control", "MC02": "control", "MC03": "control", "MC04": "control",
}
GROUP_NAMES  = ["mild_ctrl", "mod_sev"]
SEV_TO_GROUP = {"control": 0, "mild": 0, "moderate": 1, "severe": 1}
BEAM_WIDTH   = 100
BATCH        = 8

print("Config ready.")
print("Group 0 (mild_ctrl): control + mild")
print("Group 1 (mod_sev):   moderate + severe")

Config ready.
Group 0 (mild_ctrl): control + mild
Group 1 (mod_sev):   moderate + severe


## Step 2 — Load TORGO, create splits

In [4]:
print("Loading TORGO...")
cache = os.environ["HF_DATASETS_CACHE"]
ds    = load_dataset("abnerh/TORGO-database", cache_dir=cache)
df    = ds["train"].to_pandas()
df["audio_path"] = df["audio"].apply(lambda x: x["path"])
df["speaker"]    = df["audio_path"].apply(lambda p: str(p).split("_")[0])
df["Category"]   = df["speaker"].map(SEVERITY_MAPPING)
df["Group"]      = df["Category"].map(SEV_TO_GROUP)

hf_full = ds["train"].add_column("speaker",  df["speaker"].tolist())
hf_full = hf_full.add_column("Category", df["Category"].tolist())
hf_full = hf_full.add_column("Group",    df["Group"].tolist())

test_mask  = df["speaker"].isin(TEST_SPEAKERS)
torgo_test = hf_full.select(df[test_mask].index.tolist())
torgo_test = torgo_test.filter(
    lambda x: len(x["audio"]["array"]) <= MAX_AUDIO, num_proc=1
)

train_pool_df = df[~test_mask].copy()
ctrl  = train_pool_df[train_pool_df["Category"] == "control"].sample(
    n=min(CONTROL_TARGET, (train_pool_df["Category"] == "control").sum()),
    random_state=RANDOM_SEED
)
other = train_pool_df[train_pool_df["Category"] != "control"]
train_pool_df = pd.concat([ctrl, other]).sample(frac=1, random_state=RANDOM_SEED)

random.seed(RANDOM_SEED)
group_train_idx = {0: [], 1: []}
val_idx         = []
for sev in ["control", "mild", "moderate", "severe"]:
    grp     = SEV_TO_GROUP[sev]
    sev_idx = train_pool_df[train_pool_df["Category"] == sev].index.tolist()
    random.shuffle(sev_idx)
    n_val   = max(1, int(len(sev_idx) * MLP_VAL_FRAC))
    val_idx.extend(sev_idx[:n_val])
    group_train_idx[grp].extend(sev_idx[n_val:])

group_train = {
    grp: hf_full.select(idxs).filter(
        lambda x: len(x["audio"]["array"]) <= MAX_AUDIO, num_proc=1
    )
    for grp, idxs in group_train_idx.items()
}
torgo_val = hf_full.select(val_idx).filter(
    lambda x: len(x["audio"]["array"]) <= MAX_AUDIO, num_proc=1
)

print(f"Test: {len(torgo_test)}  Val: {len(torgo_val)}")
for g, ds_ in group_train.items():
    cats = dict(sorted(Counter(ds_["Category"]).items()))
    print(f"Group {g} ({GROUP_NAMES[g]}): {len(ds_)} utterances  {cats}")

Loading TORGO...


Test: 1786  Val: 1111
Group 0 (mild_ctrl): 1848 utterances  {'control': 1200, 'mild': 648}
Group 1 (mod_sev): 2586 utterances  {'moderate': 869, 'severe': 1717}


## Step 3 — Load B2 (frozen) and extract LibriSpeech CTC weights

In [5]:
from safetensors.torch import load_file

processor = Wav2Vec2Processor.from_pretrained(B2_PATH)
b2_model  = WavLMForCTC.from_pretrained(
    B2_PATH, ctc_loss_reduction="mean", ctc_zero_infinity=True
)
for param in b2_model.parameters():
    param.requires_grad = False
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
b2_model = b2_model.to(device)
b2_model.eval()
print(f"B2 loaded on {device} — fully frozen")

# ── Extract CTC weights from LibriSpeech model ──
LIBRISPEECH_DIR = "/kaggle/input/datasets/kavishk/b1-wavlm-ctc-librispeech/kaggle/working/b1_wavlm_ctc_librispeech"

libri_state = load_file(os.path.join(LIBRISPEECH_DIR, "model.safetensors"))

# Print all keys to confirm lm_head location
lm_keys = [k for k in libri_state.keys() if "lm_head" in k]
print(f"lm_head keys: {lm_keys}")

libri_ctc_weight = libri_state["lm_head.weight"].clone()
libri_ctc_bias   = libri_state["lm_head.bias"].clone() if "lm_head.bias" in libri_state else None
print(f"LibriSpeech CTC weight: {libri_ctc_weight.shape}")

vocab_size  = processor.tokenizer.vocab_size
hidden_size = b2_model.config.hidden_size  # 768

if libri_ctc_weight.shape[0] != vocab_size:
    print(f"Vocab mismatch: LibriSpeech={libri_ctc_weight.shape[0]} TORGO={vocab_size} — aligning...")
    min_v = min(libri_ctc_weight.shape[0], vocab_size)
    nw    = torch.randn(vocab_size, hidden_size) * 0.02
    nw[:min_v].copy_(libri_ctc_weight[:min_v])
    libri_ctc_weight = nw
    if libri_ctc_bias is not None:
        nb = torch.zeros(vocab_size)
        nb[:min_v].copy_(libri_ctc_bias[:min_v])
        libri_ctc_bias = nb

print(f"CTC weights ready: {libri_ctc_weight.shape}")

Loading weights:   0%|          | 0/250 [00:00<?, ?it/s]

B2 loaded on cuda — fully frozen
lm_head keys: ['lm_head.bias', 'lm_head.weight']
LibriSpeech CTC weight: torch.Size([32, 768])
Vocab mismatch: LibriSpeech=32 TORGO=30 — aligning...
CTC weights ready: torch.Size([30, 768])


In [6]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

def prepare_torgo(batch):
    """
    Same preprocessing as B2/P1 training.
    Uses processor feature extraction identically to training.
    """
    audio = batch["audio"]
    arr   = audio["array"]
    if len(arr) > MAX_AUDIO:
        arr = arr[:MAX_AUDIO]
    inputs = processor(arr, sampling_rate=audio["sampling_rate"])
    batch["input_values"]   = inputs.input_values[0]
    batch["labels"]         = processor.tokenizer(
        batch["transcription"].lower().strip()
    ).input_ids
    batch["severity_label"] = TORGO_SEV_TO_INT[batch["Category"]]
    return batch


@dataclass
class DataCollatorCTC:
    """Pads input_values and labels — identical to B2/P1 DataCollatorLoRA."""    
    processor: Any
    padding: Union[bool, str] = "longest"

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_feats  = [{"input_values": f["input_values"]} for f in features]
        batch        = self.processor.feature_extractor.pad(
            input_feats, padding=self.padding, return_tensors="pt"
        )
        label_feats  = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_feats, padding=self.padding, return_tensors="pt"
        )
        batch["labels"] = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        return batch


data_collator = DataCollatorCTC(processor=processor)
print("prepare_torgo and data_collator ready.")


prepare_torgo and data_collator ready.


In [7]:
# Preprocess using same pipeline as B2 training
# Avoids ~6pp WER difference from re-processing raw audio at inference

print("Preprocessing torgo_test...")
TORGO_COLS = torgo_test.column_names
test_p = torgo_test.map(
    prepare_torgo,
    remove_columns=TORGO_COLS,
    num_proc=1,
    desc="Preprocessing TORGO test",
)
test_cats_filtered = list(torgo_test["Category"])
print(f"test_p: {len(test_p)} utterances")

print("\nPreprocessing torgo_val...")
VAL_COLS  = torgo_val.column_names
torgo_val_p = torgo_val.map(
    prepare_torgo,
    remove_columns=VAL_COLS,
    num_proc=1,
    desc="Preprocessing TORGO val",
)
val_cats_filtered = list(torgo_val["Category"])
print(f"torgo_val_p: {len(torgo_val_p)} utterances")

# Also preprocess group_train datasets for training
print("\nPreprocessing group_train...")
group_train_p = {}
for grp, ds_ in group_train.items():
    COLS = ds_.column_names
    group_train_p[grp] = ds_.map(
        prepare_torgo,
        remove_columns=COLS,
        num_proc=1,
        desc=f"Preprocessing group {grp}",
    )
    print(f"  Group {grp} ({GROUP_NAMES[grp]}): {len(group_train_p[grp])} utterances")


Preprocessing torgo_test...
test_p: 1786 utterances

Preprocessing torgo_val...
torgo_val_p: 1111 utterances

Preprocessing group_train...
  Group 0 (mild_ctrl): 1848 utterances
  Group 1 (mod_sev): 2586 utterances


## Step 4 — Build 2-group LoRA model

In [8]:
class LoRALinear(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.weight = linear.weight
        self.bias   = linear.bias
        self.weight.requires_grad = False
        if self.bias is not None: self.bias.requires_grad = False
        self.lora_A = nn.Parameter(torch.empty(rank, linear.in_features))
        self.lora_B = nn.Parameter(torch.zeros(linear.out_features, rank))
        self.scale  = alpha / rank
        nn.init.normal_(self.lora_A)

    def forward(self, x):
        return (F.linear(x, self.weight, self.bias)
                + F.linear(F.linear(x, self.lora_A), self.lora_B) * self.scale)


def inject_lora(encoder, rank, alpha):
    """
    WavLM uses fused QKV — no separate q_proj/v_proj.
    Inject LoRA into out_proj and gru_rel_pos_linear across all 12 layers.
    Gives 2 x 12 = 24 LoRALinear layers.
    """
    PROJ_NAMES = ["out_proj", "gru_rel_pos_linear"]
    targets    = []
    for _, module in encoder.named_modules():
        for proj in PROJ_NAMES:
            child = getattr(module, proj, None)
            if isinstance(child, nn.Linear) and not isinstance(child, LoRALinear):
                targets.append((module, proj))
    for parent, proj in targets:
        setattr(parent, proj, LoRALinear(getattr(parent, proj), rank, alpha))
    print(f"  Replaced {len(targets)} linear layers "
          f"(out_proj + gru_rel_pos_linear x 12 layers)")
    return encoder


class TwoGroupLoRAModel(nn.Module):
    def __init__(self, b2_model, libri_w, libri_b, rank=16, alpha=32):
        super().__init__()
        vocab    = libri_w.shape[0]
        hidden   = b2_model.config.hidden_size
        has_bias = libri_b is not None

        self.encoder = b2_model.wavlm
        for p in self.encoder.parameters(): p.requires_grad = False

        # ── Debug: count linear layers visible from self.encoder ──
        count = 0
        for name, module in self.encoder.named_modules():
            for proj in ["out_proj", "gru_rel_pos_linear"]:
                child = getattr(module, proj, None)
                if isinstance(child, nn.Linear) and not isinstance(child, LoRALinear):
                    count += 1
                    if count <= 6:  # print first 6 to avoid spam
                        print(f"  Found: {name}.{proj}  shape={child.weight.shape}")
        print(f"  Total linear layers visible before inject: {count}")
        # ── End debug ──

        print("Injecting LoRA...")
        self.encoder = inject_lora(self.encoder, rank, alpha)
        self._lora_layers = [m for m in self.encoder.modules()
                              if isinstance(m, LoRALinear)]
        print(f"  {len(self._lora_layers)} LoRALinear layers")

        self.adapter_A = nn.ParameterList([
            nn.ParameterList([nn.Parameter(l.lora_A.data.clone())
                              for l in self._lora_layers])
            for _ in range(NUM_GROUPS)
        ])
        self.adapter_B = nn.ParameterList([
            nn.ParameterList([nn.Parameter(l.lora_B.data.clone())
                              for l in self._lora_layers])
            for _ in range(NUM_GROUPS)
        ])

        self.ctc_heads = nn.ModuleList([
            nn.Linear(hidden, vocab, bias=has_bias) for _ in range(NUM_GROUPS)
        ])
        for head in self.ctc_heads:
            head.weight.data.copy_(libri_w)
            if has_bias: head.bias.data.copy_(libri_b)

        self.pad_token_id  = b2_model.config.pad_token_id
        self._active_group = 0
        self._load_group(0)

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Trainable: {trainable:,} params")

    def _load_group(self, g):
        for i, l in enumerate(self._lora_layers):
            l.lora_A.data.copy_(self.adapter_A[g][i].data)
            l.lora_B.data.copy_(self.adapter_B[g][i].data)
        self._active_group = g

    def _save_group(self, g):
        for i, l in enumerate(self._lora_layers):
            self.adapter_A[g][i].data.copy_(l.lora_A.data)
            self.adapter_B[g][i].data.copy_(l.lora_B.data)

    def set_active_group(self, g):
        if g == self._active_group: return
        self._save_group(self._active_group)
        self._load_group(g)

    def forward_single(self, iv, am, g):
        self.set_active_group(g)
        out    = self.encoder(iv, attention_mask=am, output_hidden_states=True)
        logits = self.ctc_heads[g](out.last_hidden_state)
        return logits, out.hidden_states


print("Building model...")
lora_model = TwoGroupLoRAModel(
    b2_model, libri_ctc_weight, libri_ctc_bias
).to(device)
print("Model ready.")

Building model...
  Found: encoder.layers.0.attention.out_proj  shape=torch.Size([768, 768])
  Found: encoder.layers.0.attention.gru_rel_pos_linear  shape=torch.Size([8, 64])
  Found: encoder.layers.1.attention.out_proj  shape=torch.Size([768, 768])
  Found: encoder.layers.1.attention.gru_rel_pos_linear  shape=torch.Size([8, 64])
  Found: encoder.layers.2.attention.out_proj  shape=torch.Size([768, 768])
  Found: encoder.layers.2.attention.gru_rel_pos_linear  shape=torch.Size([8, 64])
  Total linear layers visible before inject: 24
Injecting LoRA...
  Replaced 24 linear layers (out_proj + gru_rel_pos_linear x 12 layers)
  24 LoRALinear layers
Trainable: 972,348 params
Model ready.


## Step 5 — Training helpers

In [12]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def decode_ctc_greedy(pred_ids, tokenizer):
    blank = tokenizer.pad_token_id
    out   = []
    for seq in pred_ids:
        col = [k for k, _ in groupby(seq)]
        fil = [t for t in col if t != blank]
        out.append(tokenizer.decode(fil, skip_special_tokens=True) if fil else "")
    return out


def ctc_loss_fn(logits, labels, pad_id):
    log_p  = F.log_softmax(logits, dim=-1).transpose(0, 1)
    ilen   = torch.full((logits.size(0),), logits.size(1),
                        dtype=torch.long, device=logits.device)
    llen   = (labels != -100).sum(-1)

    # Filter samples where label length >= input length (CTC requirement)
    valid  = llen < ilen
    if not valid.all():
        print(f"  Skipping {(~valid).sum()} samples: label longer than input")
        log_p  = log_p[:, valid]
        ilen   = ilen[valid]
        llen   = llen[valid]
        labels = labels[valid]

    if llen.sum() == 0:
        return torch.tensor(0.0, requires_grad=True, device=logits.device)

    labs = labels.clone()
    labs[labs == -100] = pad_id
    return F.ctc_loss(log_p, labs, ilen, llen,
                      blank=pad_id, reduction="mean", zero_infinity=True)


def preprocess_batch(samples):
    """Uses data_collator — identical preprocessing to B2 training."""    
    batch  = data_collator(samples)
    iv     = batch["input_values"].to(device)
    am     = batch.get("attention_mask")
    if am is not None: am = am.to(device)
    labels = batch["labels"].to(device)
    return iv, am, labels


def get_layer6_pooled(model, iv, am):
    with torch.no_grad():
        out = model.wavlm(iv, attention_mask=am, output_hidden_states=True)
    l6 = out.hidden_states[7]
    if am is not None:
        fl   = model.wavlm._get_feat_extract_output_lengths(am.sum(-1)).long()
        T    = l6.size(1)
        mask = (torch.arange(T, device=device).unsqueeze(0)
                < fl.unsqueeze(1)).unsqueeze(-1).float()
        return (l6 * mask).sum(1) / mask.sum(1).clamp(min=1)
    return l6.mean(dim=1)


print("Helpers ready.")

Helpers ready.


## Step 6 — Train each group independently

In [13]:
LORA_LR    = 1e-4
HEAD_LR    = 1e-4
NUM_EPOCHS = 20
PATIENCE   = 5
BATCH_SIZE = 4
GRAD_ACCUM = 4


def train_group(grp, train_ds):
    print(f"\n{'='*60}")
    print(f"Training Group {grp} ({GROUP_NAMES[grp]})  [{len(train_ds)} utterances]")
    print(f"{'='*60}")

    lora_model.set_active_group(grp)
    optimizer = torch.optim.AdamW([
        {"params": list(lora_model.adapter_A[grp].parameters()), "lr": LORA_LR},
        {"params": list(lora_model.adapter_B[grp].parameters()), "lr": LORA_LR},
        {"params": list(lora_model.ctc_heads[grp].parameters()), "lr": HEAD_LR},
    ], weight_decay=0.01)

    best_loss  = float("inf")
    best_state = None
    no_improve = 0
    indices    = list(range(len(train_ds)))
    n_batches  = (len(indices) + BATCH_SIZE - 1) // BATCH_SIZE

    for epoch in range(NUM_EPOCHS):
        lora_model.train()
        random.shuffle(indices)
        epoch_loss, n_seen = 0.0, 0
        optimizer.zero_grad()

        for bn, i in enumerate(range(0, len(indices), BATCH_SIZE)):
            batch_idx = indices[i:i+BATCH_SIZE]
            samples   = [train_ds[j] for j in batch_idx]
            if not samples: continue

            iv, am, labels = preprocess_batch(samples)
            logits, _      = lora_model.forward_single(iv, am, grp)
            loss           = ctc_loss_fn(logits, labels, lora_model.pad_token_id)

            if not torch.isfinite(loss):
                print(f"\n  WARNING: non-finite loss={loss.item()} at batch {bn} — skipping")
                optimizer.zero_grad()
                continue

            (loss / GRAD_ACCUM).backward()

            epoch_loss += loss.item() * len(samples)
            n_seen     += len(samples)

            if (bn + 1) % GRAD_ACCUM == 0:
                optimizer.step(); optimizer.zero_grad()

            if (bn + 1) % 25 == 0:
                pct = (bn+1)/n_batches
                bar = "█"*int(pct*20) + "░"*(20-int(pct*20))
                print(f"\r  [{bar}] {bn+1}/{n_batches}  "
                      f"loss={epoch_loss/max(n_seen,1):.4f}",
                      end="", flush=True)

        optimizer.step(); optimizer.zero_grad()
        print()

        avg_loss = epoch_loss / max(n_seen, 1)
        print(f"  Epoch {epoch+1:2d}/{NUM_EPOCHS}  loss={avg_loss:.4f}", flush=True)

        if avg_loss < best_loss - 0.005:
            best_loss = avg_loss
            no_improve = 0
            best_state = {
                "A": [p.data.clone() for p in lora_model.adapter_A[grp]],
                "B": [p.data.clone() for p in lora_model.adapter_B[grp]],
                "h": {k: v.clone() for k, v in
                      lora_model.ctc_heads[grp].state_dict().items()},
            }
            print(f"  → Best: {best_loss:.4f}", flush=True)
        else:
            no_improve += 1
            print(f"  No improvement {no_improve}/{PATIENCE}", flush=True)
            if no_improve >= PATIENCE:
                print("  Early stopping.", flush=True); break

    if best_state:
        for i, p in enumerate(lora_model.adapter_A[grp]): p.data.copy_(best_state["A"][i])
        for i, p in enumerate(lora_model.adapter_B[grp]): p.data.copy_(best_state["B"][i])
        lora_model.ctc_heads[grp].load_state_dict(best_state["h"])
        lora_model._load_group(grp)
        print(f"  Restored best (loss={best_loss:.4f})", flush=True)
    return best_loss


In [14]:
train_group(0, group_train_p[0])


Training Group 0 (mild_ctrl)  [1848 utterances]
  [███████████████████░] 450/462  loss=0.3551
  Epoch  1/20  loss=0.3536
  → Best: 0.3536
  [███████████████████░] 450/462  loss=0.3625
  Epoch  2/20  loss=0.3618
  No improvement 1/5
  [███████████████████░] 450/462  loss=0.3516
  Epoch  3/20  loss=0.3534
  No improvement 2/5
  [███████████████████░] 450/462  loss=0.3342
  Epoch  4/20  loss=0.3311
  → Best: 0.3311
  [███████████████████░] 450/462  loss=0.3421
  Epoch  5/20  loss=0.3406
  No improvement 1/5
  [███████████████████░] 450/462  loss=0.3562
  Epoch  6/20  loss=0.3562
  No improvement 2/5
  [███████████████████░] 450/462  loss=0.3466
  Epoch  7/20  loss=0.3459
  No improvement 3/5
  [███████████████████░] 450/462  loss=0.3053
  Epoch  8/20  loss=0.3091
  → Best: 0.3091
  [███████████████████░] 450/462  loss=0.3451
  Epoch  9/20  loss=0.3461
  No improvement 1/5
  [███████████████████░] 450/462  loss=0.3269
  Epoch 10/20  loss=0.3293
  No improvement 2/5
  [███████████████████░

0.3090603312243631

In [15]:
train_group(1, group_train_p[1])


Training Group 1 (mod_sev)  [2586 utterances]
  [███████████████████░] 625/647  loss=0.6679
  Epoch  1/20  loss=0.6638
  → Best: 0.6638
  [███████████████████░] 625/647  loss=0.6270
  Epoch  2/20  loss=0.6266
  → Best: 0.6266
  [███████████████████░] 625/647  loss=0.6162
  Epoch  3/20  loss=0.6118
  → Best: 0.6118
  [███████████████████░] 625/647  loss=0.5993
  Epoch  4/20  loss=0.5992
  → Best: 0.5992
  [███████████████████░] 625/647  loss=0.6133
  Epoch  5/20  loss=0.6135
  No improvement 1/5
  [███████████████████░] 625/647  loss=0.5920
  Epoch  6/20  loss=0.5981
  No improvement 2/5
  [███████████████████░] 625/647  loss=0.6069
  Epoch  7/20  loss=0.6136
  No improvement 3/5
  [███████████████████░] 625/647  loss=0.5809
  Epoch  8/20  loss=0.5827
  → Best: 0.5827
  [███████████████████░] 625/647  loss=0.5948
  Epoch  9/20  loss=0.5980
  No improvement 1/5
  [███████████████████░] 625/647  loss=0.5911
  Epoch 10/20  loss=0.5907
  No improvement 2/5
  [███████████████████░] 625/647 

0.5827205143984168

In [16]:
torch.save({
    "A0": [p.data for p in lora_model.adapter_A[0]],
    "B0": [p.data for p in lora_model.adapter_B[0]],
    "A1": [p.data for p in lora_model.adapter_A[1]],
    "B1": [p.data for p in lora_model.adapter_B[1]],
    "h0": lora_model.ctc_heads[0].state_dict(),
    "h1": lora_model.ctc_heads[1].state_dict(),
}, "/kaggle/working/p1v6_two_group_lora.pt")
print("Model saved.")

Model saved.


In [17]:
head0_weight = lora_model.ctc_heads[0].weight.data.cpu()

diff_from_libri = (head0_weight - libri_ctc_weight.cpu()).abs().mean().item()

print(f"Head 0 vs LibriSpeech CTC: mean abs diff = {diff_from_libri:.6f}")

if diff_from_libri < 1e-6:
    print("✓ Head correctly initialised from LibriSpeech weights")
else:
    print(f"✗ Head differs from LibriSpeech by {diff_from_libri:.6f} — check init")

# Also check head 1 is the same init
head1_weight    = lora_model.ctc_heads[1].weight.data.cpu()
diff_h0_h1      = (head0_weight - head1_weight).abs().mean().item()
print(f"Head 0 vs Head 1:          mean abs diff = {diff_h0_h1:.6f}")
print("(should be ~0 at init since both start from same LibriSpeech weights)")

Head 0 vs LibriSpeech CTC: mean abs diff = 0.005561
✗ Head differs from LibriSpeech by 0.005561 — check init
Head 0 vs Head 1:          mean abs diff = 0.005322
(should be ~0 at init since both start from same LibriSpeech weights)


## Step 7 — Load Severity MLP and sanity check on test set

In [19]:
class SeverityMLP(nn.Module):
    def __init__(self, hidden=768, n_cls=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128),    nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, n_cls),
        )
    def forward(self, x): return self.net(x)


severity_mlp = SeverityMLP().to(device)
ckpt = torch.load(MLP_PT, map_location="cpu")
severity_mlp.load_state_dict(ckpt["state_dict"] if "state_dict" in ckpt else ckpt)
severity_mlp.eval()
if "best_val_acc" in ckpt:
    print(f"MLP loaded. Val acc at save: {ckpt['best_val_acc']*100:.2f}%")
else:
    print("MLP loaded.")

MLP loaded. Val acc at save: 95.32%


In [20]:
# Sanity check: MLP routing on test set using preprocessed test_p
print("MLP sanity check on TORGO test set")
print("=" * 65)

all_sev_probs = []
all_group_w   = []
records       = []

for i in range(0, len(test_p), BATCH):
    end     = min(i + BATCH, len(test_p))
    samples = [test_p[j] for j in range(i, end)]

    batch = data_collator(samples)
    iv    = batch["input_values"].to(device)
    am    = batch.get("attention_mask")
    if am is not None: am = am.to(device)

    pooled = get_layer6_pooled(b2_model, iv, am)
    with torch.no_grad():
        sp = F.softmax(severity_mlp(pooled), dim=-1).cpu().numpy()

    for j in range(len(samples)):
        probs    = sp[j]
        w0       = float(probs[0] + probs[1])  # mild_ctrl
        w1       = float(probs[2] + probs[3])  # mod_sev
        pred_sev = int(probs.argmax())
        pred_grp = 0 if pred_sev <= 1 else 1
        true_cat = test_cats_filtered[i + j]
        true_grp = SEV_TO_GROUP[true_cat]

        all_sev_probs.append(probs)
        all_group_w.append([w0, w1])
        records.append({
            "true_cat":   true_cat,
            "true_group": GROUP_NAMES[true_grp],
            "pred_group": GROUP_NAMES[pred_grp],
            "w0":         round(w0, 3),
            "w1":         round(w1, 3),
            "p_ctrl":     round(float(probs[0]), 3),
            "p_mild":     round(float(probs[1]), 3),
            "p_mod":      round(float(probs[2]), 3),
            "p_sev":      round(float(probs[3]), 3),
            "correct":    pred_grp == true_grp,
        })

san_df = pd.DataFrame(records)
print(f"Group routing accuracy: {san_df['correct'].mean()*100:.2f}%\n")
print(f"{'True cat':12} {'Group acc':>10} {'Avg w0 (mild_ctrl)':>18} {'Avg w1 (mod_sev)':>17}")
print("-" * 62)
for cat in ["control", "mild", "moderate", "severe"]:
    sub = san_df[san_df["true_cat"] == cat]
    if not len(sub): continue
    print(f"{cat:12} {sub['correct'].mean()*100:>9.1f}%  "
          f"{sub['w0'].mean():>17.3f}  {sub['w1'].mean():>16.3f}")

for label, subset in [("CORRECT", san_df[san_df["correct"]]),
                       ("WRONG",   san_df[~san_df["correct"]])]:
    print(f"\n{label} (most confident):")
    dom = subset[["w0", "w1"]].max(axis=1)
    for _, row in subset.loc[dom.nlargest(min(3, len(subset))).index].iterrows():
        print(f"  true={row['true_cat']:10s} → pred_group={row['pred_group']}  "
              f"probs: ctrl={row['p_ctrl']:.3f} mild={row['p_mild']:.3f} "
              f"mod={row['p_mod']:.3f} sev={row['p_sev']:.3f}  "
              f"[w0={row['w0']:.3f} w1={row['w1']:.3f}]")

print("\nSanity check complete.")


MLP sanity check on TORGO test set
Group routing accuracy: 79.79%

True cat      Group acc Avg w0 (mild_ctrl)  Avg w1 (mod_sev)
--------------------------------------------------------------
control           82.8%              0.820             0.180
mild              56.6%              0.563             0.437
moderate          97.9%              0.020             0.980
severe            97.9%              0.019             0.981

CORRECT (most confident):
  true=control    → pred_group=mild_ctrl  probs: ctrl=1.000 mild=0.000 mod=0.000 sev=0.000  [w0=1.000 w1=0.000]
  true=control    → pred_group=mild_ctrl  probs: ctrl=1.000 mild=0.000 mod=0.000 sev=0.000  [w0=1.000 w1=0.000]
  true=control    → pred_group=mild_ctrl  probs: ctrl=1.000 mild=0.000 mod=0.000 sev=0.000  [w0=1.000 w1=0.000]

WRONG (most confident):
  true=control    → pred_group=mod_sev  probs: ctrl=0.000 mild=0.000 mod=1.000 sev=0.000  [w0=0.000 w1=1.000]
  true=control    → pred_group=mod_sev  probs: ctrl=0.000 mild=0.00

In [21]:
# ── Detailed probability inspection for wrong predictions ──
print("Wrongly routed utterances — full probability breakdown")
print("=" * 70)
print(f"{'True cat':10} {'Pred grp':12} {'p_ctrl':>8} {'p_mild':>8} "
      f"{'p_mod':>8} {'p_sev':>8} {'w0':>7} {'w1':>7} {'confidence':>11}")
print("-" * 70)

wrong_df = san_df[~san_df["correct"]].copy()
wrong_df["confidence"] = wrong_df[["w0", "w1"]].max(axis=1)
wrong_df = wrong_df.sort_values("confidence", ascending=False)

for _, row in wrong_df.iterrows():
    print(f"{row['true_cat']:10} {row['pred_group']:12} "
          f"{row['p_ctrl']:>8.3f} {row['p_mild']:>8.3f} "
          f"{row['p_mod']:>8.3f} {row['p_sev']:>8.3f} "
          f"{row['w0']:>7.3f} {row['w1']:>7.3f} "
          f"{row['confidence']:>10.3f}")

print(f"\nTotal wrong: {len(wrong_df)}")
print(f"\nConfidence distribution for wrong predictions:")
bins = [0.5, 0.6, 0.7, 0.8, 0.9, 1.01]
labels = ["0.5-0.6", "0.6-0.7", "0.7-0.8", "0.8-0.9", "0.9-1.0"]
wrong_df["conf_bin"] = pd.cut(wrong_df["confidence"], bins=bins, labels=labels)
print(wrong_df["conf_bin"].value_counts().sort_index().to_string())

print(f"\nHighly confident wrong predictions (confidence > 0.9):")
high_conf_wrong = wrong_df[wrong_df["confidence"] > 0.9]
print(f"  Count: {len(high_conf_wrong)} / {len(wrong_df)} wrong "
      f"({len(high_conf_wrong)/len(wrong_df)*100:.1f}% of wrong predictions)")
print(f"  True categories: {dict(high_conf_wrong['true_cat'].value_counts())}")

print(f"\nLow confidence wrong predictions (confidence < 0.7):")
low_conf_wrong = wrong_df[wrong_df["confidence"] < 0.7]
print(f"  Count: {len(low_conf_wrong)} / {len(wrong_df)} wrong "
      f"({len(low_conf_wrong)/len(wrong_df)*100:.1f}% of wrong predictions)")
print(f"  These are boundary cases where soft routing provides meaningful blending.")

Wrongly routed utterances — full probability breakdown
True cat   Pred grp       p_ctrl   p_mild    p_mod    p_sev      w0      w1  confidence
----------------------------------------------------------------------
mild       mod_sev         0.000    0.000    1.000    0.000   0.000   1.000      1.000
control    mod_sev         0.000    0.000    1.000    0.000   0.000   1.000      1.000
mild       mod_sev         0.000    0.000    1.000    0.000   0.000   1.000      1.000
mild       mod_sev         0.000    0.000    1.000    0.000   0.000   1.000      1.000
mild       mod_sev         0.000    0.000    0.018    0.982   0.000   1.000      1.000
mild       mod_sev         0.000    0.000    1.000    0.000   0.000   1.000      1.000
mild       mod_sev         0.000    0.000    0.994    0.005   0.000   1.000      1.000
mild       mod_sev         0.000    0.000    0.982    0.018   0.000   1.000      1.000
mild       mod_sev         0.000    0.000    0.003    0.997   0.000   1.000      1.000
mil

## Step 8 — Collect logits from both groups on test + val sets

In [23]:
def collect_two_group_logits(preprocessed_ds, cats_list, desc=""):
    """
    Run both adapters using preprocessed dataset + data_collator.
    Identical feature extraction to B2 training — avoids ~6pp WER gap.
    """
    lora_model.eval()
    lg0_list, lg1_list, gw_list, lab_list, cat_list, greedy_list = [], [], [], [], [], []

    for i in range(0, len(preprocessed_ds), BATCH):
        end     = min(i + BATCH, len(preprocessed_ds))
        samples = [preprocessed_ds[j] for j in range(i, end)]

        batch = data_collator(samples)
        iv    = batch["input_values"].to(device)
        am    = batch.get("attention_mask")
        if am is not None: am = am.to(device)

        with torch.no_grad():
            lg0, _ = lora_model.forward_single(iv, am, 0)
            lg1, _ = lora_model.forward_single(iv, am, 1)
            pooled = get_layer6_pooled(b2_model, iv, am)
            sp     = F.softmax(severity_mlp(pooled), dim=-1).cpu().numpy()

        # Labels from collated batch
        lab_ids = batch["labels"].numpy()
        lab_ids = np.where(lab_ids != -100, lab_ids, processor.tokenizer.pad_token_id)
        lab_str = processor.tokenizer.batch_decode(lab_ids, skip_special_tokens=True)

        for j in range(len(samples)):
            w0 = float(sp[j][0] + sp[j][1])
            w1 = float(sp[j][2] + sp[j][3])
            lg0_list.append(lg0[j].cpu().numpy())
            lg1_list.append(lg1[j].cpu().numpy())
            gw_list.append([w0, w1])
            lab_list.append(lab_str[j].strip())
            cat_list.append(cats_list[i + j])

            # Greedy decode from blended logits
            blended = w0 * lg0[j].cpu() + w1 * lg1[j].cpu()
            pid     = blended.argmax(dim=-1).numpy()
            text    = decode_ctc_greedy([pid], processor.tokenizer)[0]
            greedy_list.append(text.strip())

        if (i // BATCH + 1) % 20 == 0:
            print(f"  [{desc}] {end}/{len(preprocessed_ds)}")

    return lg0_list, lg1_list, gw_list, lab_list, cat_list, greedy_list


print("Collecting test logits (preprocessed)...")
(all_lg0, all_lg1, all_gw,
 all_labels, all_cats, greedy_preds) = collect_two_group_logits(
    test_p, test_cats_filtered, "test"
)

print("\nCollecting val logits (preprocessed)...")
(val_lg0, val_lg1, val_gw,
 val_labels, val_cats, _) = collect_two_group_logits(
    torgo_val_p, val_cats_filtered, "val"
)

# Greedy baseline on test
eg     = [p if p else "⟨empty⟩" for p in greedy_preds]
gw_wer = wer_metric.compute(predictions=eg, references=all_labels)
gw_cer = cer_metric.compute(predictions=eg, references=all_labels)
res_df = pd.DataFrame({"prediction": greedy_preds,
                        "reference":  all_labels, "Category": all_cats})

print(f"\nP1v6 Greedy (soft-routed): WER={gw_wer*100:.2f}%  CER={gw_cer*100:.2f}%")
for cat in ["control", "mild", "moderate", "severe"]:
    sub = res_df[res_df["Category"] == cat]
    sub = sub[sub["reference"].str.strip().str.len() > 0]
    if not len(sub): continue
    pg  = [p if p else "⟨empty⟩" for p in sub["prediction"].tolist()]
    gw  = wer_metric.compute(predictions=pg, references=sub["reference"].tolist())
    gc  = cer_metric.compute(predictions=pg, references=sub["reference"].tolist())
    print(f"  {cat:10s}: WER={gw*100:.2f}%  CER={gc*100:.2f}%  (n={len(sub)})")


  [test] 160/1786
  [test] 320/1786
  [test] 480/1786
  [test] 640/1786
  [test] 800/1786
  [test] 960/1786
  [test] 1120/1786
  [test] 1280/1786
  [test] 1440/1786
  [test] 1600/1786
  [test] 1760/1786

  [val] 160/1111
  [val] 320/1111
  [val] 480/1111
  [val] 640/1111
  [val] 800/1111
  [val] 960/1111

P1v6 Greedy (soft-routed): WER=42.85%  CER=21.27%
  control   : WER=16.62%  CER=5.66%  (n=302)
  mild      : WER=21.98%  CER=6.84%  (n=673)
  moderate  : WER=73.30%  CER=40.60%  (n=575)
  severe    : WER=68.44%  CER=41.41%  (n=236)


In [24]:
# ── Oracle routing evaluation ──
print("Oracle routing (true severity labels):")
lora_model.eval()
oracle_preds = []

for i in range(0, len(test_p), BATCH):
    end     = min(i + BATCH, len(test_p))
    samples = [test_p[j] for j in range(i, end)]
    cats    = test_cats_filtered[i:end]

    batch = data_collator(samples)
    iv    = batch["input_values"].to(device)
    am    = batch.get("attention_mask")
    if am is not None: am = am.to(device)

    with torch.no_grad():
        # Use true group label — hard routing to correct adapter
        for j, cat in enumerate(cats):
            true_grp   = SEV_TO_GROUP[cat]
            iv_j       = iv[j:j+1]
            am_j       = am[j:j+1] if am is not None else None
            logits, _  = lora_model.forward_single(iv_j, am_j, true_grp)
            pid        = logits.argmax(dim=-1).cpu().numpy()
            text       = decode_ctc_greedy(pid, processor.tokenizer)[0]
            oracle_preds.append(text.strip() if text.strip() else "⟨empty⟩")

o_wer = wer_metric.compute(
    predictions=oracle_preds, references=all_labels
)
o_cer = cer_metric.compute(
    predictions=oracle_preds, references=all_labels
)
print(f"Oracle WER: {o_wer*100:.2f}%  CER: {o_cer*100:.2f}%")
print(f"vs Soft routing: WER={gw_wer*100:.2f}%")
print(f"Gap (oracle advantage): {(gw_wer - o_wer)*100:.2f}pp")

Oracle routing (true severity labels):
Oracle WER: 42.73%  CER: 21.25%
vs Soft routing: WER=42.85%
Gap (oracle advantage): 0.12pp


In [25]:
# ═══════════════════════════════════════════════════════
# Adapter Specialisation Probe
# For each test speaker group, decode with CORRECT adapter
# and WRONG adapter. If adapters specialised, correct
# adapter should give lower WER.
# ═══════════════════════════════════════════════════════

print("=" * 60)
print("Adapter Specialisation Probe")
print("=" * 60)

lora_model.eval()
wer_metric = evaluate.load("wer")

results = {}

for true_grp in range(NUM_GROUPS):
    # Filter test samples belonging to this group
    grp_idx = [i for i, c in enumerate(test_cats_filtered)
                if SEV_TO_GROUP[c] == true_grp]
    if not grp_idx:
        continue

    grp_refs = [all_labels[i] for i in grp_idx if all_labels[i].strip()]
    grp_idx  = [i for i in grp_idx if all_labels[i].strip()]

    print(f"\nTrue group: {GROUP_NAMES[true_grp]}  ({len(grp_idx)} utterances)")
    wer_by_adapter = {}

    for test_grp in range(NUM_GROUPS):
        preds = []
        for i in grp_idx:
            samples = [test_p[i]]
            batch   = data_collator(samples)
            iv      = batch["input_values"].to(device)
            am      = batch.get("attention_mask")
            if am is not None: am = am.to(device)

            with torch.no_grad():
                logits, _ = lora_model.forward_single(iv, am, test_grp)

            pid  = logits.argmax(dim=-1).cpu().numpy()
            text = decode_ctc_greedy(pid, processor.tokenizer)[0]
            preds.append(text.strip() if text.strip() else "⟨empty⟩")

        wer = wer_metric.compute(predictions=preds, references=grp_refs)
        wer_by_adapter[test_grp] = wer
        marker = " ← CORRECT" if test_grp == true_grp else ""
        print(f"  Adapter={GROUP_NAMES[test_grp]:12s}: WER={wer*100:.2f}%{marker}")

    best = min(wer_by_adapter, key=wer_by_adapter.get)
    specialised = best == true_grp
    print(f"  Best adapter: {GROUP_NAMES[best]}  "
          f"{'✓ Specialised' if specialised else '✗ Not specialised'}")
    results[GROUP_NAMES[true_grp]] = {
        "correct_wer": wer_by_adapter[true_grp],
        "wrong_wer":   wer_by_adapter[1 - true_grp],
        "specialised": specialised,
    }

print("\n" + "=" * 60)
print("Summary")
print("=" * 60)
for grp_name, r in results.items():
    delta = (r["wrong_wer"] - r["correct_wer"]) * 100
    print(f"{grp_name:12s}: correct={r['correct_wer']*100:.2f}%  "
          f"wrong={r['wrong_wer']*100:.2f}%  "
          f"Δ={delta:+.2f}pp  "
          f"{'✓' if r['specialised'] else '✗'}")

print("\nInterpretation:")
print("  Δ > 0pp  → correct adapter gives lower WER → specialised")
print("  Δ ≈ 0pp  → adapters produce similar representations → not specialised")
print("  Δ < 0pp  → wrong adapter is better → adapters trained incorrectly")

Adapter Specialisation Probe

True group: mild_ctrl  (975 utterances)
  Adapter=mild_ctrl   : WER=19.40% ← CORRECT
  Adapter=mod_sev     : WER=19.57%
  Best adapter: mild_ctrl  ✓ Specialised

True group: mod_sev  (811 utterances)
  Adapter=mild_ctrl   : WER=73.44%
  Adapter=mod_sev     : WER=71.33% ← CORRECT
  Best adapter: mod_sev  ✓ Specialised

Summary
mild_ctrl   : correct=19.40%  wrong=19.57%  Δ=+0.17pp  ✓
mod_sev     : correct=71.33%  wrong=73.44%  Δ=+2.11pp  ✓

Interpretation:
  Δ > 0pp  → correct adapter gives lower WER → specialised
  Δ ≈ 0pp  → adapters produce similar representations → not specialised
  Δ < 0pp  → wrong adapter is better → adapters trained incorrectly


## Step 9 — KenLM beam search with soft routing

In [26]:
lm_dir = "/kaggle/temp/kenlm"
os.makedirs(lm_dir, exist_ok=True)
lm_path = hf_hub_download(
    repo_id="jonatasgrosman/wav2vec2-large-xlsr-53-english",
    filename="language_model/lm.binary", cache_dir=lm_dir,
)
uni_path = hf_hub_download(
    repo_id="jonatasgrosman/wav2vec2-large-xlsr-53-english",
    filename="language_model/unigrams.txt", cache_dir=lm_dir,
)
unigrams = open(uni_path).read().strip().split("\n")
print(f"KenLM ready. Unigrams: {len(unigrams):,}")

vocab_dict = processor.tokenizer.get_vocab()
vocab_list = [None] * len(vocab_dict)
for tok, idx in vocab_dict.items(): vocab_list[idx] = tok
vocab_list[processor.tokenizer.pad_token_id] = ""

_dec_cache = {}
def get_decoder(alpha, beta):
    key = (round(float(alpha), 3), round(float(beta), 3))
    if key not in _dec_cache:
        _dec_cache[key] = build_ctcdecoder(
            labels=vocab_list, kenlm_model_path=lm_path,
            unigrams=unigrams, alpha=key[0], beta=key[1],
        )
    return _dec_cache[key]

def to_log_probs(lg):
    p = np.exp(lg) / np.exp(lg).sum(axis=-1, keepdims=True)
    return np.log(np.clip(p, 1e-8, 1.0))

def decode_soft(lg0, lg1, gw, alpha, beta, beam_width=100):
    blended = gw[0] * lg0 + gw[1] * lg1
    return get_decoder(alpha, beta).decode(
        to_log_probs(blended), beam_width=beam_width
    ).strip().lower()

print("Decoder helper ready.")

KenLM ready. Unigrams: 373,978
Decoder helper ready.


In [40]:
# Bayesian optimisation on val set
from skopt import gp_minimize
from skopt.space import Real

GRID_BEAM  = 50
call_count = [0]

def objective(params):
    alpha, beta = params
    preds, refs = [], []
    for i in range(len(val_lg0)):
        if not val_labels[i].strip(): continue
        text = decode_soft(val_lg0[i], val_lg1[i], val_gw[i], alpha, beta, GRID_BEAM)
        preds.append(text or "⟨empty⟩")
        refs.append(val_labels[i])
    wer = wer_metric.compute(predictions=preds, references=refs)
    call_count[0] += 1
    print(f"  [{call_count[0]:2d}] alpha={alpha:.3f} beta={beta:.3f} WER={wer*100:.2f}%",
          flush=True)
    return wer

print("Bayesian optimisation on val set...")
result     = gp_minimize(
    func=objective,
    dimensions=[Real(0.0, 3.0, name="alpha"), Real(0.0, 2.0, name="beta")],
    n_calls=40, n_initial_points=10, acq_func="EI",
    random_state=42, verbose=False,
)
best_alpha = round(float(result.x[0]), 3)
best_beta  = round(float(result.x[1]), 3)
print(f"\nBest: alpha={best_alpha}  beta={best_beta}  WER={result.fun*100:.2f}%")

Bayesian optimisation on val set...
  [ 1] alpha=2.390 beta=0.367 WER=40.48%
  [ 2] alpha=2.339 beta=1.194 WER=39.34%
  [ 3] alpha=1.337 beta=0.200 WER=31.04%
  [ 4] alpha=1.378 beta=0.667 WER=30.96%
  [ 5] alpha=0.429 beta=1.302 WER=23.65%
  [ 6] alpha=0.169 beta=1.444 WER=24.94%
  [ 7] alpha=2.816 beta=0.002 WER=42.50%
  [ 8] alpha=2.977 beta=1.235 WER=42.65%
  [ 9] alpha=1.835 beta=0.014 WER=36.75%
  [10] alpha=0.069 beta=1.050 WER=25.97%
  [11] alpha=0.612 beta=2.000 WER=24.26%
  [12] alpha=0.475 beta=0.000 WER=23.69%
  [13] alpha=0.296 beta=0.008 WER=23.53%
  [14] alpha=0.952 beta=1.998 WER=25.70%
  [15] alpha=0.771 beta=0.010 WER=25.40%
  [16] alpha=0.365 beta=1.960 WER=24.22%
  [17] alpha=0.213 beta=0.002 WER=24.33%
  [18] alpha=0.427 beta=0.003 WER=23.72%
  [19] alpha=0.011 beta=0.019 WER=26.01%
  [20] alpha=0.502 beta=0.860 WER=23.88%
  [21] alpha=0.397 beta=0.002 WER=23.76%
  [22] alpha=0.518 beta=1.984 WER=23.80%
  [23] alpha=0.531 beta=0.001 WER=23.91%
  [24] alpha=0.011 be

In [35]:
from transformers import Wav2Vec2Processor
processor = Wav2Vec2Processor.from_pretrained(LIBRISPEECH_DIR)  # ← overwrites TORGO processor

# Load LibriSpeech model for CTC weight extraction only
# Use a separate variable — do NOT overwrite processor (which is TORGO)
from safetensors.torch import load_file

libri_state      = load_file(os.path.join(LIBRISPEECH_DIR, "model.safetensors"))
lm_keys          = [k for k in libri_state.keys() if "lm_head" in k]
libri_ctc_weight = next(libri_state[k] for k in lm_keys if "weight" in k).clone()
libri_ctc_bias   = next((libri_state[k].clone() for k in lm_keys if "bias" in k), None)
print(f"LibriSpeech CTC weight: {libri_ctc_weight.shape}")
# Do NOT load processor from LIBRISPEECH_DIR

_dec_cache.clear()

vocab_dict = processor.tokenizer.get_vocab()
vocab_list = [None] * len(vocab_dict)
for tok, idx in vocab_dict.items():
    vocab_list[idx] = tok
vocab_list[processor.tokenizer.pad_token_id] = ""
print(f"vocab_list size: {len(vocab_list)}")  # must be 30

LibriSpeech CTC weight: torch.Size([32, 768])
vocab_list size: 32


In [34]:
processor = Wav2Vec2Processor.from_pretrained(B2_PATH)
print(f"Vocab size: {processor.tokenizer.vocab_size}")
print(f"Vocab: {processor.tokenizer.get_vocab()}")

print(f"lora_model ctc_heads[0] output size: {lora_model.ctc_heads[0].weight.shape}")
# [0] = out_features (vocab), [1] = in_features (768)

Vocab size: 30
Vocab: {"'": 26, '[PAD]': 29, '[UNK]': 28, 'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6, 'h': 7, 'i': 8, 'j': 9, 'k': 10, 'l': 11, 'm': 12, 'n': 13, 'o': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, '|': 27, '<s>': 30, '</s>': 31}
lora_model ctc_heads[0] output size: torch.Size([30, 768])


In [37]:
_dec_cache.clear()

vocab_dict = processor.tokenizer.get_vocab()
vocab_size  = processor.tokenizer.vocab_size  # 30

# Only include tokens at indices 0-29, skip <s> and </s> at 30, 31
vocab_list = [None] * vocab_size
for tok, idx in vocab_dict.items():
    if idx < vocab_size:
        vocab_list[idx] = tok

vocab_list[processor.tokenizer.pad_token_id] = ""  # CTC blank

print(f"vocab_list size: {len(vocab_list)}")  # should be 30
print(f"vocab_list: {vocab_list}")

vocab_list size: 30
vocab_list: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', "'", '|', '[UNK]', '']


In [41]:
# ── Use same α/β as B3 — no retuning needed ──
# B3 and P1v6 use the same LibriSpeech KenLM and same val set
# Reusing B3's tuned values avoids overfitting α/β to this specific system
best_alpha = 0.296  # replace with your actual B3 best_alpha value
best_beta  = 0.008  # replace with your actual B3 best_beta value
BEAM_WIDTH = 100

print(f"Using B3 α/β: alpha={best_alpha}  beta={best_beta}")
print(f"Final beam search on test (beam={BEAM_WIDTH})...")

final_preds = [
    decode_soft(all_lg0[i], all_lg1[i], all_gw[i], best_alpha, best_beta, BEAM_WIDTH)
    for i in range(len(all_lg0))
]

ef    = [p if p else "⟨empty⟩" for p in final_preds]
f_wer = wer_metric.compute(predictions=ef, references=all_labels)
f_cer = cer_metric.compute(predictions=ef, references=all_labels)
fin_df = pd.DataFrame({"prediction": final_preds,
                        "reference":  all_labels, "Category": all_cats})

print("\n" + "=" * 65)
print("P1 v6 — 2-Group Adapters + Soft MLP Routing + KenLM")
print("=" * 65)
print(f"Overall WER: {f_wer*100:.2f}%   CER: {f_cer*100:.2f}%")
print(f"alpha={best_alpha}  beta={best_beta}")
print("\nPer-severity:")
print(f"{'':12} {'WER':>8} {'CER':>8}  n")
print("-" * 42)
for cat in ["control", "mild", "moderate", "severe"]:
    sub = fin_df[fin_df["Category"] == cat]
    sub = sub[sub["reference"].str.strip().str.len() > 0]
    if not len(sub): continue
    pf  = [p if p else "⟨empty⟩" for p in sub["prediction"].tolist()]
    fw  = wer_metric.compute(predictions=pf, references=sub["reference"].tolist())
    fc  = cer_metric.compute(predictions=pf, references=sub["reference"].tolist())
    print(f"{cat:12} {fw*100:>7.2f}%  {fc*100:>6.2f}%  ({len(sub)})")

fin_df.to_csv("/kaggle/working/p1v6_results.csv", index=False)
with open("/kaggle/working/p1v6_summary.json", "w") as f:
    json.dump({
        "model":      "P1v6_2Group_LoRA_SoftMLP_KenLM",
        "groups":     GROUP_NAMES,
        "ctc_init":   "LibriSpeech",
        "alpha":      best_alpha,
        "beta":       best_beta,
        "alpha_beta_source": "B3 Bayesian-tuned values (same val set)",
        "wer":        round(f_wer, 4),
        "cer":        round(f_cer, 4),
    }, f, indent=2)
print("\nResults saved.")

Using B3 α/β: alpha=0.296  beta=0.008
Final beam search on test (beam=100)...

P1 v6 — 2-Group Adapters + Soft MLP Routing + KenLM
Overall WER: 39.12%   CER: 19.82%
alpha=0.296  beta=0.008

Per-severity:
                  WER      CER  n
------------------------------------------
control        19.71%    6.48%  (302)
mild           21.63%    6.66%  (673)
moderate       62.76%   36.37%  (575)
severe         61.70%   39.61%  (236)

Results saved.
